In [1]:
import os
import json
import re
import pdfplumber
import tiktoken
from pathlib import Path

In [50]:
pdf_path = "../data/raw/pdfs"
OUTPUT_PATH = "../data/processed/wk10_chunks.json"

In [51]:
def extract_pdf_text(pdf_path):
    text_pages = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()

            if text:
                text_pages.append({
                    "page": page_num,
                    "text": text
                })

    return text_pages

In [52]:
pdf_files = list(Path(pdf_path).glob("*.pdf"))

print(pdf_files)

[WindowsPath('../data/raw/pdfs/force_laws_motion.pdf')]


In [53]:
all_pages = []

for pdf in pdf_files:
    pages = extract_pdf_text(pdf)
    all_pages.extend(pages)

print("Pages Loaded:", len(all_pages))


Pages Loaded: 13


In [77]:
def clean_text(text):

    # normalize spaces
    text = re.sub(r'\s+', ' ', text)

    # remove repeated capital noise
    text = re.sub(r'\b([A-Z])\1{2,}\b', '', text)

    # remove weird repeated OCR letters
    text = re.sub(r'[A-Z]{4,}', '', text)

    # remove page footer noise
    text = re.sub(r'Reprint\s+\d{4}-\d{2}', '', text)

    # remove excessive spaces again
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [56]:
encoding = tiktoken.get_encoding("cl100k_base")

In [57]:
def count_tokens(text):
    return len(encoding.encode(text))

In [58]:
def detect_content_type(text):

    text_lower = text.lower()

    if "example" in text_lower:
        return "worked_example"

    elif "exercise" in text_lower or "question" in text_lower:
        return "question"

    elif "=" in text or "formula" in text_lower:
        return "equation"

    else:
        return "concept"

In [59]:
def create_chunks(pages, chunk_size=250, overlap=40):

    chunks = []

    chunk_id = 0

    for page_data in pages:

        page_text = page_data["text"]
        page_num = page_data["page"]

        words = page_text.split()

        current_chunk = []

        current_tokens = 0

        for word in words:

            token_count = count_tokens(word)

            if current_tokens + token_count > chunk_size:

                chunk_text = " ".join(current_chunk)

                chunks.append({
                    "chunk_id": f"chunk_{chunk_id}",
                    "page": page_num,
                    "content_type": detect_content_type(chunk_text),
                    "text": chunk_text
                })

                overlap_words = current_chunk[-overlap:]

                current_chunk = overlap_words
                current_tokens = count_tokens(" ".join(current_chunk))

                chunk_id += 1

            current_chunk.append(word)
            current_tokens += token_count

        if current_chunk:

            chunk_text = " ".join(current_chunk)

            chunks.append({
                "chunk_id": f"chunk_{chunk_id}",
                "page": page_num,
                "content_type": detect_content_type(chunk_text),
                "text": chunk_text
            })

            chunk_id += 1

    return chunks

In [78]:
chunks = create_chunks(all_pages)

In [79]:
print(chunks[0])

{'chunk_id': 'chunk_0', 'page': 1, 'content_type': 'concept', 'text': '8 C hapter FFFFF LLLLL MMMMM OOOOORRRRRCCCCCEEEEE AAAAANNNNNDDDDD AAAAAWWWWWSSSSS OOOOOFFFFF OOOOOTTTTTIIIIIOOOOONNNNN In the previous chapter, we described the In our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. We We saw that such a motion can be uniform ordinarily experience this as a muscular effort or non-uniform. We have not yet discovered and say that we must push or hit or pull on what causes the motion. Why does the speed an object to change its state of motion. The of an object change with time? Do all motions concept of force is based on this push, hit or require a cause? If so, what is the nature of pull. Let us now ponder about a ‘force’. What this cause? In this chapter we shall make an is it? In fact, no one has seen, tasted or felt 

In [69]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=4, ensure_ascii=False)

print("Saved:", OUTPUT_PATH)

Saved: ../data/processed/wk10_chunks.json
